[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap02/cap02_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

# Da Captura ao Pixel – Amostragem, Quantização e Conectividade

Este capítulo explora como uma cena contínua se transforma em uma matriz discreta de pixels, abordando os processos de amostragem espacial, quantização de intensidades e as relações topológicas entre os elementos da imagem.

## Objetivos

Ao final deste capítulo, você será capaz de:

- Explicar o modelo físico de formação da imagem baseado em iluminação e refletância.
- Diferenciar os mecanismos da visão humana e dos sensores digitais.
- Compreender os processos de **amostragem** (discretização do espaço) e **quantização** (discretização da intensidade).
- Descrever as relações topológicas entre pixels: vizinhança, adjacência, conectividade e distâncias.
- Realizar transformações geométricas básicas (translação, rotação, escala) preservando a qualidade.
- Visualizar na prática os efeitos da variação da resolução espacial e da profundidade de bits.

## Antes de começar: *Notebooks* em Python

Este material segue o conceito de ***Literate Programming*** (Programação Literária) proposto por Donald Knuth. Para executar uma célula, pressione <kbd>Shift</kbd> + <kbd>Enter</kbd> ou clique no botão ▶️.

::: {.callout-note}
### Nota sobre o formato {.unnumbered}
Nas versões renderizadas (PDF ou HTML), o código é apresentado em blocos estáticos. A execução interativa requer o Google Colab (badge no topo) ou ambiente local com Jupyter/VSCode.
:::

## O Olho e a Câmera – Elementos da Percepção Visual

A formação de uma imagem digital começa com a captura da luz refletida pelos objetos. Tanto o olho humano quanto as câmeras digitais seguem princípios semelhantes, mas com diferenças importantes.

### Visão humana

O olho é um sistema óptico complexo: a luz atravessa a córnea, o humor aquoso, o cristalino (que ajusta o foco) e o humor vítreo até atingir a retina. Na retina, os **cones** (≈6 milhões) são responsáveis pela visão de cores (três tipos sensíveis ao vermelho, verde e azul), enquanto os **bastonetes** (≈120 milhões) operam em baixa luminosidade, fornecendo apenas intensidade. O ponto cego é a região de saída do nervo óptico, desprovida de fotorreceptores.

### Sensores digitais

Câmeras digitais utilizam sensores **CCD** ou **CMOS** que substituem a retina. Cada elemento do sensor (fotossítio) acumula carga elétrica proporcional à luz incidente. Um **filtro de Bayer** (matriz de filtros coloridos) permite capturar as três componentes de cor. Um conversor analógico-digital (ADC) quantiza a carga em um valor numérico. Diferentemente do olho, o sensor digital tem resolução espacial fixa (número de pixels) e profundidade de bits definida (ex.: 8 bits por canal).

> **Curiosidade:** A resolução efetiva da fóvea (visão central de alta definição) é equivalente a aproximadamente 120 × 120 pixels. O processamento cerebral realiza enorme pós-processamento para gerar a percepção de alta resolução.

## O Modelo Matemático da Formação da Imagem

Uma imagem pode ser modelada como o produto de duas funções:

$$
f(x,y) = i(x,y) \cdot r(x,y)
$$ {#eq-02-formacao}

onde:

- $i(x,y)$ é a **iluminação** incidente sobre a cena (energia luminosa por unidade de área). Varia lentamente no espaço.
- $r(x,y)$ é a **refletância** do objeto (fração da luz refletida). Varia rapidamente em bordas e texturas.

O processamento de imagens frequentemente tenta separar ou compensar essas componentes – por exemplo, em correção de iluminação não uniforme.

## Digitalização: Amostragem e Quantização

Para transformar uma cena contínua em uma imagem digital, dois processos são necessários.

### Amostragem – Discretização do espaço

A amostragem consiste em medir o valor da função $f(x,y)$ em pontos igualmente espaçados, formando uma matriz de $M$ linhas (altura) e $N$ colunas (largura). A **resolução espacial** é $M \times N$. Quanto maior a resolução, mais detalhes são preservados, mas maior o custo computacional e de armazenamento.

### Quantização – Discretização da intensidade

A quantização associa a cada pixel um valor numérico discreto, geralmente representado por um inteiro de $b$ bits. A **profundidade de bits** define o número de níveis de intensidade: $2^b$. Imagens em tons de cinza usam 8 bits (256 níveis); imagens coloridas usam três canais de 8 bits (24 bits no total).

::: {.callout-warning}
### Erro de quantização {.unnumbered}
A diferença entre o valor analógico real e o valor discreto atribuído manifesta-se como ruído de quantização, visível em regiões com gradiente suave quando se usam poucos bits (efeito de "posterização").
:::

### Laboratório prático: Efeitos da amostragem e quantização

Os experimentos a seguir mostram como a redução da resolução espacial (subamostragem) e da profundidade de bits degradam a qualidade visual. Use o código para explorar diferentes fatores e níveis de cinza.

In [ ]:
#| quarto-raw: true

import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Baixar morph.py se necessário
BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

In [ ]:
# Carregar imagem de exemplo (Lena)
url_lena = "https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
img_color = mm.read(url_lena)
img_gray = mm.gray(img_color)
print(f"Imagem original: {img_gray.shape}")

In [ ]:
#| label: fig-02-subamostragem
#| fig-cap: "Efeito da subamostragem (redução de resolução espacial) com fatores 1, 2, 4 e 8. A imagem é reduzida e depois redimensionada de volta ao tamanho original por vizinho mais próximo para comparação."
#| echo: true
#| output: true

def subsample_effect(image, factor):
    """Reduz a imagem por um fator e depois a redimensiona ao tamanho original (vizinho mais próximo)."""
    h, w = image.shape
    reduced = image[::factor, ::factor]
    return mm.resize(reduced, (w, h), method='nearest')

factors = [1, 2, 4, 8]
images = [subsample_effect(img_gray, f) for f in factors]
titles = [f"Original (fator {f})" if f==1 else f"Subamostragem fator {f}" for f in factors]

mm.show(images, titles=titles, cols=4, figsize=(12, 4))

In [ ]:
#| label: fig-02-quantizacao
#| fig-cap: "Efeito da redução da profundidade de bits: (a) 8 bits (256 níveis), (b) 4 bits (16 níveis), (c) 2 bits (4 níveis), (d) 1 bit (2 níveis). Perceba a posterização e perda de gradientes."
#| echo: true
#| output: true

def quantize(image, bits):
    """Reduz a imagem para 2^bits níveis de cinza."""
    levels = 2 ** bits
    # Quantização uniforme
    quantized = np.floor(image / 256 * levels) / levels * 255
    return quantized.astype(np.uint8)

bits_list = [8, 4, 2, 1]
images_quant = [quantize(img_gray, b) for b in bits_list]
titles_quant = [f"{b} bits ({2**b} níveis)" for b in bits_list]

mm.show(images_quant, titles=titles_quant, cols=4, figsize=(12, 4))

## Relações entre Pixels – Topologia da Imagem

Os pixels não são elementos isolados; suas posições relativas definem importantes conceitos para processamento.

### Vizinhança

Dado um pixel de coordenadas $(x,y)$, definem-se dois tipos principais de vizinhança (para imagens em grid retangular):

- **Vizinhança-4** (von Neumann): inclui os pixels nas posições $(x-1,y)$, $(x+1,y)$, $(x,y-1)$, $(x,y+1)$.
- **Vizinhança-8** (Moore): inclui todos os oito pixels adjacentes (acrescenta os quatro diagonais).

A escolha da vizinhança influencia operações como detecção de bordas, cálculo de gradientes e conectividade.

### Adjacência, conectividade e caminhos

Dois pixels são **adjacentes** se estão em contato segundo uma vizinhança definida **e** satisfazem um critério de valor (ex.: mesmo nível de intensidade). Uma **conectividade** define uma relação de equivalência entre pixels que formam uma região conexa. Um **caminho** é uma sequência de pixels adjacentes.

A conectividade-4 e conectividade-8 podem produzir resultados diferentes na segmentação e no cálculo de componentes conexas (etiquetagem). Por exemplo, um padrão de tabuleiro de xadrez pode ser completamente desconexo em 4-vizinhança, mas totalmente conexo em 8-vizinhança.

### Distâncias entre pixels

Várias métricas podem ser usadas para medir distância entre pixels $(x_1, y_1)$ e $(x_2, y_2)$:

| Métrica | Definição | Interpretação |
|:--------|:----------|:--------------|
| **Euclidiana** | $\sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}$ | Distância em linha reta (contínua) |
| **Manhattan (City block)** | $|x_1-x_2| + |y_1-y_2|$ | Movimentos horizontais + verticais |
| **Chebyshev (Tabuleiro)** | $\max(|x_1-x_2|, |y_1-y_2|)$ | Movimentos incluindo diagonais |

Essas distâncias são usadas em algoritmos de interpolação, transformada de distância, crescimento de regiões, etc.

::: {.callout-note}
### Exemplo prático
Para dois pixels separados por $\Delta x = 3$, $\Delta y = 4$:
- Euclidiana = 5
- Manhattan = 7
- Chebyshev = 4
:::

## Armazenamento de Imagens

Uma imagem digital pode ser armazenada em diversos formatos, cada um com características próprias:

| Formato | Características | Uso típico |
|:--------|:----------------|:-------------|
| **BMP** | Não comprimido (ou compressão simples) | Windows, aplicações legado |
| **PNG** | Compressão sem perdas (lossless), suporta transparência | Web, imagens com bordas nítidas |
| **JPEG** | Compressão com perdas (lossy), boa para fotografias | Fotos, imagens médicas (uso moderado) |
| **TIFF** | Suporta múltiplas camadas e compressão variada | Editoração, arquivamento |
| **RAW** | Dados brutos do sensor, sem processamento | Fotografia profissional |

Os metadados de uma imagem incluem: largura, altura, profundidade de bits, tipo de codificação de cor (grayscale, RGB, CMYK), informações de calibração (ex.: resolução DPI) e metadados EXIF (data, modelo da câmera, configurações). Ao ler uma imagem com `mm.read()`, a biblioteca `morph` preserva automaticamente os metadados relevantes.

## Transformações Geométricas Básicas

As transformações geométricas alteram a posição dos pixels, mantendo os valores de intensidade. São fundamentais para alinhamento, correção de distorções e aumento de dados.

### Translação

Desloca a imagem em $t_x$ e $t_y$ pixels:

$$
\begin{cases} x' = x + t_x \\ y' = y + t_y \end{cases}
$$

Na prática, cria-se uma nova imagem maior e copia-se os pixels deslocados. Pixels que saem da área original são descartados; áreas vazias são preenchidas com 0 (preto).

### Rotação

Gira a imagem em torno da origem (ou de seu centro) por um ângulo $\theta$. A transformação direta pode produzir buracos; na prática, usa-se a **transformação inversa** (mapeamento de cada pixel da imagem destino para a origem) e interpolação (vizinho mais próximo, bilinear, bicúbica). A rotação geralmente requer redimensionamento da imagem para que nenhuma área seja cortada.

### Escala (redimensionamento)

Altera o tamanho da imagem pelos fatores $s_x$ (horizontal) e $s_y$ (vertical). O redimensionamento exige interpolação:

- **Vizinho mais próximo:** rápido, mas produz efeito de blocos.
- **Bilinear:** suaviza melhor, é um bom compromisso.
- **Bicúbica:** mais suave, usada em softwares profissionais.

A biblioteca `morph` oferece funções como `mm.resize(img, (new_w, new_h), method='bilinear')` e `mm.rotate(img, angle, center=None, interp='bilinear')`.

In [ ]:
#| label: fig-02-rotacao
#| fig-cap: "Exemplo de rotação da imagem Lena em 30° e 45° usando interpolação bilinear."
#| echo: true
#| output: true

img_rot30 = mm.rotate(img_gray, 30, interp='bilinear')
img_rot45 = mm.rotate(img_gray, 45, interp='bilinear')

mm.show([img_gray, img_rot30, img_rot45],
        titles=["Original", "Rotação 30°", "Rotação 45°"],
        cols=3, figsize=(10, 4))

In [ ]:
#| label: fig-02-escala
#| fig-cap: "Redimensionamento com diferentes métodos de interpolação: vizinho mais próximo vs. bilinear."
#| echo: true
#| output: true

h, w = img_gray.shape
new_size = (w*2, h*2)  # dobrar de tamanho

img_nearest = mm.resize(img_gray, new_size, method='nearest')
img_bilinear = mm.resize(img_gray, new_size, method='bilinear')

mm.show([img_gray, img_nearest, img_bilinear],
        titles=["Original", "Vizinho mais próximo (2×)", "Bilinear (2×)"],
        cols=3, figsize=(12, 4))

## Resumo

Neste capítulo foram apresentados os fundamentos de digitalização e topologia de imagens:

- **Formação da imagem:** $f(x,y) = i(x,y) \cdot r(x,y)$.
- **Amostragem:** discretização do espaço → resolução espacial $M \times N$.
- **Quantização:** discretização da intensidade → profundidade de bits $b$.
- **Relações topológicas:** vizinhança-4, vizinhança-8, conectividade, distâncias (Euclidiana, Manhattan, Chebyshev).
- **Transformações geométricas:** translação, rotação, escala (com interpolação bilinear ou vizinho mais próximo).
- **Formatos de arquivo:** BMP, PNG, JPEG, TIFF, RAW; cada um com diferentes compromissos entre qualidade e tamanho.

O Capítulo 3 abordará **operações espaciais** como convolução, filtragem e morfologia matemática (erosão, dilatação).

## 🤖 Uso do NotebookLM como Tutor Complementar

Nesta edição, incentivamos o uso do **NotebookLM** como ferramenta complementar de aprendizagem. Essa ferramenta de IA utiliza exclusivamente os documentos fornecidos pelo autor como base de conhecimento, garantindo respostas coerentes com o conteúdo do livro.

Para cada capítulo, preparamos um projeto específico na plataforma. Para uma experiência de estudo ampliada, utilize o acesso abaixo:

::: {.callout-important appearance="default" icon=false}
### 🎓 Estude com o Tutor Inteligente {.unnumbered}

Para interagir com o conteúdo deste capítulo, acesse o link a seguir. O ambiente contém materiais didáticos em diferentes formatos, gerados a partir do **PDF** do capítulo. Na plataforma, explore especialmente as opções **Guia de Estudo** e **Conversa** para aprofundar sua compreensão.

[🚀 ACESSAR NOTEBOOKLM: CAPÍTULO 02](https://notebooklm.google.com/notebook/022d09fb-4dee-46b1-ad18-787987fe976d)
:::

## Lista de Exercícios

1. **(15%)** Explique, com suas próprias palavras, a diferença entre **amostragem** e **quantização**. Dê um exemplo concreto de cada uma no contexto de uma imagem digital.

2. **(15%)** Considere uma imagem com resolução espacial de 1024 × 768 pixels e profundidade de 24 bits (8 bits por canal RGB). Calcule o tamanho total não comprimido da imagem em bytes e em megabytes.

3. **(20%)** Utilizando o código do laboratório, modifique o fator de subamostragem para 3 e para 6. Descreva visualmente o que ocorre com as bordas dos objetos. O que é o efeito de *aliasing*?

4. **(20%)** Para a imagem em tons de cinza, aplique quantização com 3 bits (8 níveis) e 5 bits (32 níveis). Compare os resultados e explique por que 5 bits já pode ser considerado suficiente para muitas aplicações.

5. **(15%)** Dados dois pixels $A=(10,20)$ e $B=(15,25)$, calcule as distâncias Euclidiana, Manhattan e Chebyshev entre eles.

6. **(15%)** Usando a função `mm.rotate`, gire a imagem Lena em ângulos de 90°, 180° e 270° com interpolação bilinear. Compare com a rotação usando `method='nearest'`. Em quais situações a interpolação vizinho mais próximo ainda é útil?

## Referências do Capítulo {.unnumbered}

A fundamentação teórica deste capítulo baseia-se nas seguintes obras:

* @gonzalez2018digital para os conceitos de amostragem, quantização e relações entre pixels.
* @szeliski2022 para transformações geométricas e conectividade.
* @bradski2008learning para a implementação prática com OpenCV e `morph.py`.
* @otsu1979 para o método de limiarização automática (citado no contexto de processamento).